In [ ]:
!pip install -q sentence-transformers scikit-learn pandas torch


In [ ]:
import pandas as pd
from sentence_transformers import CrossEncoder


In [ ]:
from datasets import load_dataset

dataset = load_dataset("microsoft/ms_marco", "v1.1")


In [ ]:
print(dataset)


In [ ]:
print(dataset["train"][0])


In [ ]:
dataset["train"][0]["passages"]


In [ ]:
rows = []

for item in dataset["train"]:
    query = item["query"]
    answers = item["answers"]

    for p in item["passages"]["passage_text"]:
        rows.append({
            "query": query,
            "finalpassage": p,
            "answers": answers
        })


In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
print(df.head())


In [ ]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)


In [ ]:
train_df.to_csv("train.csv", index=False)
valid_df.to_csv("valid.csv", index=False)


In [ ]:
print(pd.read_csv("train.csv").head())
print(pd.read_csv("valid.csv").head())



In [ ]:
rows = []

for item in dataset["train"]:
    query = item["query"]
    answers = item["answers"]

    passages = item["passages"]["passage_text"]
    labels = item["passages"]["is_selected"]

    for p, l in zip(passages, labels):
        rows.append({
            "query": query,
            "finalpassage": p,
            "answers": answers,
            "label": int(l)
        })


In [ ]:
df = pd.DataFrame(rows)
print(df["label"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


In [ ]:
def clean_text_columns(df, query_col, text_col):
    df = df.dropna(subset=[query_col, text_col])
    df[query_col] = df[query_col].astype(str)
    df[text_col] = df[text_col].astype(str)
    return df


In [ ]:
QUERY_COL = "query"
TEXT_COL = "finalpassage"
ANSWER_COL = "answers"
LABEL_COL = "label"   # only if using labels


In [ ]:
valid_df = clean_text_columns(valid_df, QUERY_COL, TEXT_COL)


In [ ]:
# Column names
QUERY_COL = "query"
TEXT_COL = "finalpassage"
ANSWER_COL = "answers"
LABEL_COL = "label"

# Cleaning function
def clean_text_columns(df, query_col, text_col):
    df = df.dropna(subset=[query_col, text_col])
    df[query_col] = df[query_col].astype(str)
    df[text_col] = df[text_col].astype(str)
    return df


In [ ]:
print(valid_df.head())
print(valid_df.isnull().sum())


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


In [ ]:
def rerank(df):
    pairs = list(zip(df[QUERY_COL], df[TEXT_COL]))
    scores = reranker.predict(pairs)

    df = df.copy()
    df["rerank_score"] = scores

    return df.sort_values("rerank_score", ascending=False)


In [ ]:
from sentence_transformers import CrossEncoder



In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda"
)


In [ ]:
import pandas as pd
from sentence_transformers import CrossEncoder

QUERY_COL = "query"
TEXT_COL = "finalpassage"
ANSWER_COL = "answers"
LABEL_COL = "label"


In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda"
)


In [ ]:
def rerank(df):
    pairs = list(zip(df[QUERY_COL], df[TEXT_COL]))
    scores = reranker.predict(pairs, batch_size=32)

    df = df.copy()
    df["rerank_score"] = scores
    return df.sort_values("rerank_score", ascending=False)


In [ ]:
reranked_valid = rerank(valid_df)


In [ ]:
print("Reranking completed!")
print(reranked_valid.shape)


In [ ]:
assert "reranked_valid" in globals(), "Please run reranked_valid = rerank(valid_df) first"
print("reranked_valid shape:", reranked_valid.shape)


In [ ]:
reranked_valid[[QUERY_COL, TEXT_COL, "rerank_score"]].head(10)


In [ ]:
reranked_by_query = (
    reranked_valid
    .groupby(QUERY_COL, group_keys=False)
    .apply(lambda x: x.sort_values("rerank_score", ascending=False))
)


In [ ]:
TOP_K = 5
topk_results = reranked_by_query.groupby(QUERY_COL).head(TOP_K)



In [ ]:
sample_query = topk_results[QUERY_COL].iloc[0]

topk_results[topk_results[QUERY_COL] == sample_query][
    [TEXT_COL, "rerank_score"]
]


In [ ]:
HAS_LABELS = LABEL_COL in topk_results.columns
print("Labels available:", HAS_LABELS)


In [ ]:
if HAS_LABELS:
    precision_k = (
        topk_results
        .groupby(QUERY_COL)[LABEL_COL]
        .mean()
        .mean()
    )
    print(f"Precision@{TOP_K}:", precision_k)


In [ ]:
def reciprocal_rank(df):
    for idx, row in enumerate(df.itertuples(), start=1):
        if getattr(row, LABEL_COL, 0) == 1:
            return 1 / idx
    return 0


In [ ]:
if HAS_LABELS:
    mrr = (
        reranked_by_query
        .groupby(QUERY_COL)
        .apply(reciprocal_rank)
        .mean()
    )
    print("MRR:", mrr)


In [ ]:
import numpy as np

def ndcg_at_k(df, k):
    rels = df[LABEL_COL].tolist()
    dcg = sum(
        rel / np.log2(i + 2)
        for i, rel in enumerate(rels[:k])
    )
    ideal = sum(
        rel / np.log2(i + 2)
        for i, rel in enumerate(sorted(rels, reverse=True)[:k])
    )
    return dcg / ideal if ideal > 0 else 0


In [ ]:
if HAS_LABELS:
    ndcg = (
        reranked_by_query
        .groupby(QUERY_COL)
        .apply(lambda x: ndcg_at_k(x, TOP_K))
        .mean()
    )
    print(f"NDCG@{TOP_K}:", ndcg)


In [ ]:
topk_results.to_csv("final_reranked_results.csv", index=False)
print("Saved: final_reranked_results.csv")


In [ ]:
print("===== FINAL SUMMARY =====")
print("Total pairs:", len(reranked_valid))
print("Queries:", reranked_valid[QUERY_COL].nunique())
print("Average rerank score:", reranked_valid["rerank_score"].mean())

if HAS_LABELS:
    print(f"Precision@{TOP_K}:", precision_k)
    print("MRR:", mrr)
    print(f"NDCG@{TOP_K}:", ndcg)
